
# Q-MARL Multi-Seed Evaluation tren MATE

Notebook nay dung de chay Q-MARL voi nhieu seed va tong hop variance thuc te cua thuat toan.

Muc tieu:
- chay it nhat 3 seed
- train moi seed theo cung mot cau hinh
- danh gia reward, coverage, transport va delivered cargoes theo checkpoint
- tong hop mean/std theo env steps


In [ ]:

from pathlib import Path
import json
import time

import gymnasium as gym
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import mate
from mate.agents import GreedyTargetAgent
from mate.wrappers.discrete_action_spaces import DiscreteCamera
from q_marl import QMARL, QMARLConfig


In [ ]:

RUN_NAME = "qmarl_multiseed_mate_4v4_9"
CONFIG_NAME = "MATE-4v4-9.yaml"
SEEDS = [11, 22, 33]

CHECKPOINT_DIR = Path("checkpoints")
RESULTS_DIR = Path("results")
MULTISEED_DIR = RESULTS_DIR / RUN_NAME
MULTISEED_DIR.mkdir(parents=True, exist_ok=True)
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

TOTAL_ENV_STEPS = 60_000
TRAIN_CHUNK_STEPS = 10_000
EVAL_EVERY_STEPS = 10_000
EVAL_EPISODES = 3
EVAL_MAX_STEPS = 400

BASE_CONFIG = dict(
    env_config=CONFIG_NAME,
    num_envs=4,
    rollout_length=128,
    n_epochs=2,
    num_mini_batches=2,
    graph_depth=3,
    edge_dim=10,
    action_levels=5,
    lr=0.003,
    entropy_coef=0.01,
    normalize_rewards=True,
    critic_loss="huber",
    critic_extra_steps=1,
    device="cpu",
    render_mode="rgb_array",
)


In [ ]:

def make_camera_env(render_mode="rgb_array"):
    base_env = gym.make(
        "MultiAgentTracking-v0",
        config=CONFIG_NAME,
        render_mode=render_mode,
    )
    return mate.MultiCamera.make(base_env, target_agent=GreedyTargetAgent())


def discrete_action_to_continuous(env, action_indices, levels=5):
    grid = DiscreteCamera.discrete_action_grid(levels=levels).astype(np.float32)
    action_high = np.asarray(
        [env.unwrapped.camera_rotation_step, env.unwrapped.camera_zooming_step],
        dtype=np.float32,
    )
    action_indices = np.asarray(action_indices, dtype=np.int64).reshape(env.unwrapped.num_cameras)
    return action_high * grid[action_indices]


def evaluate_current_agent(agent, episodes=EVAL_EPISODES, max_steps=EVAL_MAX_STEPS):
    env = make_camera_env()
    rows = []
    try:
        for episode_idx in range(1, episodes + 1):
            observation, _ = env.reset(seed=episode_idx)
            episode_reward = 0.0
            coverage_rate = float(getattr(env.unwrapped, "coverage_rate", np.nan))
            transport_rate = float(getattr(env.unwrapped, "mean_transport_rate", np.nan))
            delivered_cargoes = float(getattr(env.unwrapped, "num_delivered_cargoes", 0.0))

            for _ in range(max_steps):
                action_idx = agent.predict_env(env, observation, deterministic=True)
                action = discrete_action_to_continuous(env, action_idx)
                observation, reward, terminated, truncated, info = env.step(action)
                episode_reward += float(np.mean(reward)) if not np.isscalar(reward) else float(reward)
                done = bool(terminated or truncated)
                info0 = info[0] if isinstance(info, list) and len(info) > 0 and isinstance(info[0], dict) else (info if isinstance(info, dict) else {})
                coverage_rate = float(info0.get("coverage_rate", getattr(env.unwrapped, "coverage_rate", np.nan)))
                transport_rate = float(info0.get("mean_transport_rate", getattr(env.unwrapped, "mean_transport_rate", np.nan)))
                delivered_cargoes = float(info0.get("num_delivered_cargoes", getattr(env.unwrapped, "num_delivered_cargoes", 0.0)))
                if done:
                    break

            rows.append({
                "episode_reward": episode_reward,
                "coverage_rate": coverage_rate,
                "transport_rate": transport_rate,
                "delivered_cargoes": delivered_cargoes,
            })
    finally:
        env.close()

    df = pd.DataFrame(rows)
    return {
        "eval_episode_reward": float(df["episode_reward"].mean()),
        "eval_coverage_rate": float(df["coverage_rate"].mean()),
        "eval_transport_rate": float(df["transport_rate"].mean()),
        "eval_delivered_cargoes": float(df["delivered_cargoes"].mean()),
    }


In [ ]:

all_rows = []

for seed in SEEDS:
    run_name = f"{RUN_NAME}_seed_{seed}"
    agent = QMARL(config=QMARLConfig(seed=seed, **BASE_CONFIG))
    last_eval_step = 0
    num_chunks = (TOTAL_ENV_STEPS + TRAIN_CHUNK_STEPS - 1) // TRAIN_CHUNK_STEPS

    try:
        for chunk_idx in range(1, num_chunks + 1):
            target_steps = min(agent.total_env_steps + TRAIN_CHUNK_STEPS, TOTAL_ENV_STEPS)
            stats = agent.learn(total_env_steps=target_steps)
            env_steps = int(stats["total_env_steps"])

            row = {
                "seed": seed,
                "chunk": chunk_idx,
                "env_steps": env_steps,
                "actor_loss": float(stats["actor_loss"]),
                "critic_loss": float(stats["critic_loss"]),
                "entropy": float(stats["entropy"]),
                "mean_delta": float(stats["mean_delta"]),
            }

            if (env_steps - last_eval_step >= EVAL_EVERY_STEPS) or (env_steps >= TOTAL_ENV_STEPS):
                row.update(evaluate_current_agent(agent))
                last_eval_step = env_steps

            all_rows.append(row)
            print(seed, env_steps, {k: row.get(k) for k in ['actor_loss','critic_loss','entropy','eval_episode_reward','eval_coverage_rate','eval_transport_rate']})
    finally:
        agent.close()

multi_df = pd.DataFrame(all_rows)
multi_path = MULTISEED_DIR / "multiseed_history.csv"
multi_df.to_csv(multi_path, index=False)
print(multi_path)


In [ ]:

multi_df = pd.read_csv(MULTISEED_DIR / "multiseed_history.csv")
eval_df = multi_df.dropna(subset=["eval_episode_reward"]).copy()
summary_df = eval_df.groupby("env_steps").agg(
    reward_mean=("eval_episode_reward", "mean"),
    reward_std=("eval_episode_reward", "std"),
    coverage_mean=("eval_coverage_rate", "mean"),
    coverage_std=("eval_coverage_rate", "std"),
    transport_mean=("eval_transport_rate", "mean"),
    transport_std=("eval_transport_rate", "std"),
    cargo_mean=("eval_delivered_cargoes", "mean"),
    cargo_std=("eval_delivered_cargoes", "std"),
).reset_index()
summary_df


In [ ]:

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.ravel()

plots = [
    ("reward_mean", "reward_std", "Eval Reward"),
    ("coverage_mean", "coverage_std", "Coverage"),
    ("transport_mean", "transport_std", "Transport"),
    ("cargo_mean", "cargo_std", "Delivered Cargoes"),
]

for ax, (mean_col, std_col, title) in zip(axes, plots):
    ax.plot(summary_df["env_steps"], summary_df[mean_col], marker="o")
    std = summary_df[std_col].fillna(0.0)
    ax.fill_between(summary_df["env_steps"], summary_df[mean_col] - std, summary_df[mean_col] + std, alpha=0.2)
    ax.set_title(title)
    ax.set_xlabel("Env Steps")
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()
